# Sensitivity Analysis
Plots out sensitivity analysis graph

## Read in the data

In [ ]:
%cd /home/ian/Projects/pi-cropopt

# Note: this is a first swing at performance measuring. Look to step1e_pTest.ipynb for the newer version.

import matplotlib.pyplot as plt
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import numpy as np
import pandas as pd

import itertools 
from multiprocessing import Pool

run_file_names = [
                  "/rdata/ian/pico/paperRuns/sensitivityAnalysis/global_run_table_1988.pkl" 
                  ]

years = [ 1988 ]

dm_range_lower = 20
dm_range_upper = 30
dm_range_str = f"{dm_range_lower}to{dm_range_upper}"

global_pf_file_names = [f"/rdata/ian/pico/paperRuns/finalRuns/global_pf_{year}.pkl" for year in years]

full_run_tabs = [pd.read_pickle(run_file_name) for run_file_name in run_file_names]

full_run_tab = pd.concat(full_run_tabs, axis=0)

global_pfs = [pd.read_pickle(file) for file in global_pf_file_names]

global_pf = pd.concat(global_pfs, axis=0)


del global_pfs
del global_pf
del full_run_tabs



In [ ]:

# Drop the variable columns
to_drop = ["var%d" % i for i in range(0, 67)]
full_run_tab.drop(to_drop, axis=1, inplace=True)


## Configuration Definition
What configurations do we want to measure against our baseline? 

In [ ]:
pop_size =  [60,       60,    60,    60,    60,    60,    60,    60,    60,    40,    60,    80,   100,   120]
eta =       [4,         4,     4,     4,     2,     4,     6,     8,    10,     4,     4,     4,     4,   4  ]
tau =       [5,        10,    15,    20,    10,    10,    10,    10,    10,    10,    10,    10,    10,   10 ]

analysis =  ["tau", "tau", "tau", "tau", "eta", "eta", "eta", "eta", "eta", "pop", "pop", "pop", "pop", "pop"]

configurations = [
    np.all([
        full_run_tab['pop_size']  == pop_size[c],
        full_run_tab['eta']  == eta[c],
        full_run_tab['tau']  == tau[c], 
        full_run_tab['analysis'] == analysis[c]
        ], axis=0) 
    for c in range(len(pop_size))  

]


## Performance metric functions


#### Prefered Final Solution 

In [ ]:
def determine_preferred_for_config(df): 

    years = set(df["year"])
    algorithms = set(df["algorithm"])
    dm_ranges = set(df["DM_range"])
    dm_range = dm_ranges.pop()

    ideal_point = np.array([25, 10700])

    max_irr = 400
    max_yield = 12000

    if len(years) != 1: 
        raise ValueError("Bad table given. Should only be one year. Got:" + str(years))

    if len(algorithms) != 1 or len(dm_ranges): 
        raise ValueError("Bad algorithm or dm_range found")

    F = df.loc[:,('yield','irr_total')].values

    # Choose preferred solution
    [dm_range_lower, dm_range_upper] = dm_range.split("to")
    dm_range_lower = int(dm_range_lower)
    dm_range_upper = int(dm_range_upper)

    # What points are within bound? These are the first choice
    mask = np.logical_and(F[:, 1] >= dm_range_lower, F[:, 1] <= dm_range_upper)

    F_inbound = F[mask,:]

    # Are there any solutions within bounds? 
    if F_inbound.shape[0] != 0:
        # If so, choose the solution with the highest yield
        chosen_yield = max(F_inbound[:,0])
    else: 
        # Otherwise, just choose the highest yield 
        chosen_yield = max(F[:,0])

    # Pull out the chosen solution's objective
    F = df.loc[df["yield"] == chosen_yield, ('yield', 'irr_total')].values 

    df['preferred_yield'] = F[0][0]
    df['preferred_irr_total'] = F[0][1]

    F_norm = F / np.array([max_yield, max_irr])
    ideal_point_norm = ideal_point / np.array([max_yield, max_irr])


    df['performance'] = np.linalg.norm(F_norm - ideal_point_norm)

    return df.loc[:,('algorithm',  'gen', 'DM_range', 'preferred_yield','preferred_irr_total', 'year', 'performance')]


def determine_preferred_solutions(tab):

    max_gen = max(tab['gen'])

    # Filter all but the final generations 
    tab = tab[tab["gen"] == max_gen]

    # Determine the preferred solution for each configuration  
    summary = tab.groupby(by=['pop_size', 'eta', 'tau', 'run']).apply(determine_preferred_for_config, include_groups=False)

    return summary.drop_duplicates()


### Run the analysis 
The goal being that we compress each run into a given metric. So at the end we should have one row per run, with the given metrics calculated for that run

In [ ]:
def analyzeResults(args):
    year = args[0]
    config = args[1]
    print("Processing year and config %d-%d..." % (year, id(config)))

    result = determine_preferred_solutions(full_run_tab[config])
 
    print("Completed year and config %d-%d..." % (year, id(config)))

    return result

run_summary  = None 

run_summaries = []

args = list(itertools.product(* [years, configurations]))

len(list(args))
with Pool(20) as p: 
    run_summaries = p.map(analyzeResults, args)

run_summary = pd.concat(run_summaries)
run_summary


## Sensitivity Analysis plot 
### Tau plot

In [ ]:
current_pop_size = 60
current_eta = 4

tau_summary = run_summary.loc[current_pop_size,current_eta,:]
display(tau_summary.shape)

axes = tau_summary.boxplot(by="tau", column="performance", figsize=(5, 6));

fig = axes.get_figure()
fig.suptitle('')

axes.set_title("Sensitivity Analysis of $𝜏$ on Performance")
axes.set_xlabel("$𝜏$")
axes.set_ylabel("Performance (normalized distance to ideal point)")
plt.gcf().subplots_adjust(right=0.85)

plt.savefig("/home/ian/Downloads/tau_sensitivity.eps",format="eps")

### Eta plot

In [ ]:
current_tau = 10
current_population = 60

eta_summary = run_summary.loc[current_pop_size,:, current_tau]
display(eta_summary.shape)

axes = eta_summary.boxplot(by="eta", column="performance", figsize=(5, 6))

fig = axes.get_figure()
fig.suptitle('')

axes.set_title("Sensitivity Analysis of $\eta$ on Performance")
axes.set_xlabel("$\eta$")
axes.set_ylabel("Performance (normalized distance to ideal point)")

plt.gcf().subplots_adjust(right=0.85)

plt.savefig("/home/ian/Downloads/eta_sensitivity.eps",format="eps")

## Population Plot 

In [ ]:
current_tau = 10
current_eta = 4 

pop_summary = run_summary.loc[:,current_eta,current_tau]

display(pop_summary.shape)

axes = pop_summary.boxplot(by="pop_size", column="performance", figsize=(5, 6))


# Sighhh... matplotlib
fig = axes.get_figure()
fig.suptitle('')

axes.set_title("Sensitivity Analysis of Population Size on Performance")
axes.set_xlabel("Population Size")
axes.set_ylabel("Performance (normalized distance to ideal point)")

plt.gcf().subplots_adjust(right=0.85)

plt.savefig("/home/ian/Downloads/pop_sensitivity.eps",format="eps")